Q2：
- Why we need both: Because Precision/recall shows a systematic measurement. We can track performance over time. While spot-audit can let us know the issues in the real world qualitatively.
- If we only evaluate on a gold set, we might face distribution shift. Because the gold set is static, the model could  get high precision or recall on old data, but it will miss new data and never gets updated.
- If we only audit a small random sample, we could miss some systematic error in a specific type of documents. This can't be detected by sopt-audits.

In [15]:
# =============================================================================
# llm_human.py
# API Use + Human-in-the-Loop Extraction Tutorial
# =============================================================================
# This script demonstrates:
#   1) Defining a structured schema for event extraction (Pydantic)
#   2) Running a free local LLM (Qwen2.5-1.5B-Instruct) to extract events
#   3) Parsing and validating model outputs with fallback defaults
#   4) Generating uncertainty flags for human review
#   5) Creating an audit sheet CSV for human-in-the-loop validation
#   6) Evaluating extraction quality with precision/recall on a gold set
#
# Teaching note:
# - No user-defined functions, no if/else branching (sequential workflow)
# - Steps are explicit so students can follow and edit one piece at a time
# =============================================================================

# -----------------------------------------------------------------------------
# Setup
# -----------------------------------------------------------------------------
# Install dependencies if needed:
#   pip install pandas numpy torch transformers pydantic scikit-learn

import os
import json
import re

import numpy as np
import pandas as pd
import torch

from datetime import date
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

np.random.seed(123)
set_seed(123)

# -----------------------------------------------------------------------------
# Part 1: Define a Schema (What fields do we want to extract?)
# -----------------------------------------------------------------------------

EventType = Literal[
    "protest",
    "election",
    "policy_change",
    "violence",
    "disaster",
    "other"
]

GeoPrecision = Literal[
    "country_only",
    "admin1_or_state",
    "city_or_local",
    "unknown"
]

class EvidenceSpan(BaseModel):
    field: Literal["event_type", "date", "location", "actors", "outcome"]
    quote: str

class EventExtraction(BaseModel):
    doc_id: str

    # NOTE (teaching/demo setting):
    # For local small models, strict JSON/schema enforcement often yields empty outputs.
    # We therefore provide safe defaults so the pipeline can produce "messy but usable"
    # records while still flagging uncertainty.
    # In a production setting, tighten these requirements and fail fast.

    event_type: EventType = "other"

    event_date_iso: Optional[str] = Field(
        default=None,
        description="ISO date YYYY-MM-DD if available; otherwise null."
    )
    date_is_approximate: bool = Field(
        default=True,
        description="True if the date is estimated/inferred (e.g., 'early April')."
    )

    country: Optional[str] = None
    admin1_or_state: Optional[str] = None
    city_or_local: Optional[str] = None
    geo_precision: GeoPrecision = "unknown"

    actors: List[str] = Field(
        default_factory=list,
        description="Key actors mentioned (individuals, orgs, groups)."
    )
    outcome_summary: Optional[str] = Field(
        default=None,
        description="One-sentence outcome summary (what happened)."
    )
    extraction_confidence: float = Field(
        default=0.2, ge=0.0, le=1.0,
        description="Model self-rated confidence (0 to 1)."
    )
    uncertainty_flags: List[str] = Field(
        default_factory=list,
        description="Issues that make extraction uncertain."
    )
    evidence: List[EvidenceSpan] = Field(
        default_factory=list,
        description="Short quotes supporting each extracted field."
    )

# -----------------------------------------------------------------------------
# Part 2: Create Messy Text Inputs (Mini Corpus)
# -----------------------------------------------------------------------------
docs = [
    {"doc_id": "doc_001", "text": "Breaking: Thousands rallied in Santiago on 2026-03-14 demanding pension reform. Police reported minor clashes; 12 were arrested."},
    {"doc_id": "doc_002", "text": "On March 2nd, lawmakers passed the 'Clean Air Act' amendment in the national assembly. Environmental groups praised the vote."},
    {"doc_id": "doc_003", "text": "Election officials said voting will take place next Sunday. Turnout is expected to be high in the capital."},
    {"doc_id": "doc_004", "text": "A 6.2 magnitude earthquake struck near the coastal city overnight, damaging dozens of homes and cutting power to 40,000 residents."},
    {"doc_id": "doc_005", "text": "Witnesses described gunfire outside a nightclub late Friday; at least two people were injured, but details remain unclear."},
    {"doc_id": "doc_006", "text": "The governor announced a new curfew order effective immediately. Critics called it an overreach."},
    {"doc_id": "doc_007", "text": "Early April saw renewed demonstrations in the northern province after fuel prices rose again."},
    {"doc_id": "doc_008", "text": "Floodwaters inundated low-lying neighborhoods; emergency shelters opened at local schools, officials said."},
    {"doc_id": "doc_009", "text": "Opposition leaders met with international observers in Brussels to discuss election monitoring."},
    {"doc_id": "doc_010", "text": "Police said the suspect was arrested after a stabbing in downtown; the mayor urged calm."},
    {"doc_id": "doc_011", "text": "Parliament reversed the prior ban on rideshare apps, citing labor market flexibility."},
    {"doc_id": "doc_012", "text": "A protest was planned for tomorrow, but organizers postponed it due to severe weather warnings."},
    {"doc_id": "doc_013", "text": "Following a landslide, the ministry declared a state of emergency in two districts."},
    {"doc_id": "doc_014", "text": "The court ruling sparked demonstrations across the city center; human rights groups condemned the decision."},
    {"doc_id": "doc_015", "text": "The article mentions reforms and elections in passing but gives no clear time or place."},
]

docs_df = pd.DataFrame(docs)

print("\n------------------------------")
print("Input corpus (first 5 docs)")
print("------------------------------")
print(docs_df.head())
print("docs_df shape:", docs_df.shape)

# -----------------------------------------------------------------------------
# Part 3: Prompt Design (Schemas + Guardrails)
# -----------------------------------------------------------------------------
# We ask the model to return exactly 9 labeled key:value lines.
# This is more robust than asking for JSON from small local models,
# which often produce malformed brackets or missing fields.

system_instructions = (
    "Task: Extract ONE event record from the text.\n"
    "Return EXACTLY the following 9 lines, one per line, in the format key: value\n"
    "Use empty value if unknown.\n"
    "\n"
    "event_type: protest|election|policy_change|violence|disaster|other\n"
    "event_date_iso: YYYY-MM-DD\n"
    "date_is_approximate: true|false\n"
    "country:\n"
    "admin1_or_state:\n"
    "city_or_local:\n"
    "geo_precision: country_only|admin1_or_state|city_or_local|unknown\n"
    "actors: comma-separated list\n"
    "outcome_summary: one sentence\n"
    "\n"
    "Do not output anything else.\n"
)

print("\n------------------------------")
print("System prompt:")
print("------------------------------")
print(system_instructions)

# -----------------------------------------------------------------------------
# Part 4: Load Local LLM (Free, runs on CPU)
# -----------------------------------------------------------------------------
# Model: Qwen/Qwen2.5-1.5B-Instruct
# - Free to download from Hugging Face
# - Small enough to run on CPU (slow but works for demos)
# - Instruction-tuned: follows key:value output format reasonably well

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("\n------------------------------")
print("Loading tokenizer + model:", model_name)
print("------------------------------")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = model.to(device)

print("Device:", device)

# -----------------------------------------------------------------------------
# Part 5: Run Extraction (One Document at a Time)
# -----------------------------------------------------------------------------
# For each document:
#   1) Build the prompt
#   2) Tokenize
#   3) Generate output
#   4) Decode output text
#   5) Parse key:value lines
#   6) Validate each field, assign safe defaults if invalid
#   7) Store result

extractions = []

allowed_keys = {
    "event_type", "event_date_iso", "date_is_approximate",
    "country", "admin1_or_state", "city_or_local",
    "geo_precision", "actors", "outcome_summary"
}

print("\n------------------------------")
print("Running extraction (one doc at a time)")
print("------------------------------")

for i in range(len(docs_df)):
    doc_id = docs_df.loc[i, "doc_id"]
    text   = docs_df.loc[i, "text"]

    # --- Step 1: Build prompt ---
    prompt = (
        f"{system_instructions}\n"
        f"Document ID: {doc_id}\n"
        f"Text: {text}\n"
    )

    # --- Step 2: Tokenize ---
    inputs       = tokenizer(prompt, return_tensors="pt", truncation=True)
    input_ids    = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # --- Step 3: Generate ---
    # do_sample=False + temperature=0.0 = greedy decoding (deterministic, reproducible)
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    # --- Step 4: Decode ---
    out_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # --- Step 5: Parse key:value lines ---
    # We scan lines for "key: value" patterns and only keep recognised keys.
    # This is best-effort: malformed output is flagged, not crashed on.
    parse_ok    = True
    parse_error = ""
    parse_flags: List[str] = []

    lines = [ln.strip() for ln in out_text.splitlines() if ":" in ln]
    kv = {}
    for ln in lines:
        k, v   = ln.split(":", 1)
        k_norm = k.strip().lower()
        if k_norm in allowed_keys:
            kv[k_norm] = v.strip()

    # If nothing parsed, flag it
    if len(kv) == 0:
        parse_ok    = False
        parse_error = "no_key_value_lines_found"
        parse_flags.append("parse_failed_local_model_output")

    # --- Step 6: Validate and assign safe defaults ---

    # event_type
    event_type = (kv.get("event_type", "other") or "other").strip().lower()
    if event_type not in {"protest", "election", "policy_change", "violence", "disaster", "other"}:
        parse_flags.append("invalid_event_type_from_model")
        event_type = "other"

    # event_date_iso
    event_date_iso = (kv.get("event_date_iso", "") or "").strip() or None

    # date_is_approximate
    date_is_approx_raw = (kv.get("date_is_approximate", "") or "").strip().lower()
    if date_is_approx_raw in {"true", "false"}:
        date_is_approximate = (date_is_approx_raw == "true")
    else:
        parse_flags.append("date_is_approximate_missing_or_invalid")
        date_is_approximate = True

    # location fields
    country        = (kv.get("country", "")        or "").strip() or None
    admin1_or_state = (kv.get("admin1_or_state", "") or "").strip() or None
    city_or_local  = (kv.get("city_or_local", "")  or "").strip() or None

    # geo_precision
    geo_precision = (kv.get("geo_precision", "unknown") or "unknown").strip().lower()
    if geo_precision not in {"country_only", "admin1_or_state", "city_or_local", "unknown"}:
        parse_flags.append("invalid_geo_precision_from_model")
        geo_precision = "unknown"

    # actors
    actors_raw = (kv.get("actors", "") or "").strip()
    actors = [a.strip() for a in actors_raw.split(",") if a.strip()] if actors_raw else []

    # outcome_summary
    outcome_summary = (kv.get("outcome_summary", "") or "").strip() or None

    # --- Step 7: Build validated record ---
    extracted_obj = EventExtraction(
        doc_id=doc_id,
        event_type=event_type,
        event_date_iso=event_date_iso,
        date_is_approximate=date_is_approximate,
        country=country,
        admin1_or_state=admin1_or_state,
        city_or_local=city_or_local,
        geo_precision=geo_precision,
        actors=actors,
        outcome_summary=outcome_summary,
        extraction_confidence=0.35 if parse_ok else 0.2,
        uncertainty_flags=parse_flags,
        evidence=[]
    )

    extra_dict = extracted_obj.model_dump()

    # Attach trace fields for debugging
    extra_dict["raw_text"]              = text
    extra_dict["local_model_raw_output"] = out_text
    extra_dict["parse_ok"]             = parse_ok
    extra_dict["parse_error"]          = parse_error

    # Flatten list fields for CSV compatibility
    extra_dict["evidence_json"]          = json.dumps(extra_dict["evidence"],          ensure_ascii=False)
    extra_dict["uncertainty_flags_json"] = json.dumps(extra_dict["uncertainty_flags"], ensure_ascii=False)
    extra_dict.pop("evidence")
    extra_dict.pop("uncertainty_flags")

    extractions.append(extra_dict)
    print(f"  [{i+1:02d}/15] {doc_id} → event_type={event_type} | parse_ok={parse_ok}")

# Build DataFrame and save
extractions_df = pd.DataFrame(extractions)

print("\n------------------------------")
print("Extracted records (first 5 rows)")
print("------------------------------")
print(extractions_df[["doc_id", "event_type", "event_date_iso", "country", "parse_ok"]].head())
print("extractions_df shape:", extractions_df.shape)

os.makedirs("outputs", exist_ok=True)
extractions_df.to_csv("outputs/extractions_raw.csv", index=False)
print("Saved: outputs/extractions_raw.csv")

# Parse failure report
n_parse_fail  = (~extractions_df["parse_ok"]).sum()
pct_parse_fail = n_parse_fail / len(extractions_df) * 100
print(f"\nParse failures: {n_parse_fail} / {len(extractions_df)} ({pct_parse_fail:.1f}%)")



------------------------------
Input corpus (first 5 docs)
------------------------------
    doc_id                                               text
0  doc_001  Breaking: Thousands rallied in Santiago on 202...
1  doc_002  On March 2nd, lawmakers passed the 'Clean Air ...
2  doc_003  Election officials said voting will take place...
3  doc_004  A 6.2 magnitude earthquake struck near the coa...
4  doc_005  Witnesses described gunfire outside a nightclu...
docs_df shape: (15, 2)

------------------------------
System prompt:
------------------------------
Task: Extract ONE event record from the text.
Return EXACTLY the following 9 lines, one per line, in the format key: value
Use empty value if unknown.

event_type: protest|election|policy_change|violence|disaster|other
event_date_iso: YYYY-MM-DD
date_is_approximate: true|false
country:
admin1_or_state:
city_or_local:
geo_precision: country_only|admin1_or_state|city_or_local|unknown
actors: comma-separated list
outcome_summary: one s

Loading weights: 100%|██████████| 338/338 [00:04<00:00, 73.34it/s, Materializing param=model.norm.weight]                               


Device: cpu

------------------------------
Running extraction (one doc at a time)
------------------------------
  [01/15] doc_001 → event_type=other | parse_ok=True
  [02/15] doc_002 → event_type=policy_change | parse_ok=True
  [03/15] doc_003 → event_type=other | parse_ok=True
  [04/15] doc_004 → event_type=disaster | parse_ok=True
  [05/15] doc_005 → event_type=other | parse_ok=True
  [06/15] doc_006 → event_type=other | parse_ok=True
  [07/15] doc_007 → event_type=protest | parse_ok=True
  [08/15] doc_008 → event_type=disaster | parse_ok=True
  [09/15] doc_009 → event_type=other | parse_ok=True
  [10/15] doc_010 → event_type=other | parse_ok=True
  [11/15] doc_011 → event_type=policy_change | parse_ok=True
  [12/15] doc_012 → event_type=protest | parse_ok=True
  [13/15] doc_013 → event_type=other | parse_ok=True
  [14/15] doc_014 → event_type=protest | parse_ok=True
  [15/15] doc_015 → event_type=policy_change | parse_ok=True

------------------------------
Extracted records (firs

I use Qwen/Qwen2.5-1.5B-Instruct, and my prompt is Task: Extract ONE event record from the text.
Return EXACTLY the following 9 lines, one per line, in the format key: value
Use empty value if unknown.

event_type: protest|election|policy_change|violence|disaster|other
event_date_iso: YYYY-MM-DD
date_is_approximate: true|false
country:
admin1_or_state:
city_or_local:
geo_precision: country_only|admin1_or_state|city_or_local|unknown
actors: comma-separated list
outcome_summary: one sentence

Do not output anything else.

Document ID: doc_XXX
Text: [document text here]

In [16]:
import os
print(os.getcwd())
# -----------------------------------------------------------------------------
#  Q5. Uncertainty Flags (Automatic Triggers for Human Review)
# -----------------------------------------------------------------------------
# Each flag is a boolean column. A row with ANY flag set = needs human review.
# This is the "mechanical" part of human-in-the-loop — the system decides
# which rows are risky enough to escalate.

extractions_df["extraction_confidence"] = pd.to_numeric(
    extractions_df["extraction_confidence"], errors="coerce"
)

extractions_df["flag_parse_failed"]    = ~extractions_df["parse_ok"]
extractions_df["flag_low_confidence"]  = extractions_df["extraction_confidence"] < 0.70
extractions_df["flag_missing_date"]    = extractions_df["event_date_iso"].isna()
extractions_df["flag_missing_country"] = extractions_df["country"].isna()
extractions_df["flag_geo_unknown"]     = extractions_df["geo_precision"].isin(["unknown", "country_only"])

flag_cols = [
    "flag_parse_failed",
    "flag_low_confidence",
    "flag_missing_date",
    "flag_missing_country",
    "flag_geo_unknown"
]

# A row needs review if ANY flag is True
extractions_df["needs_human_review"] = extractions_df[flag_cols].any(axis=1)

print("\n------------------------------")
print("Review flag counts")
print("------------------------------")
print(extractions_df[flag_cols + ["needs_human_review"]].sum(numeric_only=True))

extractions_df.to_csv("outputs/extractions_with_flags.csv", index=False)
print("Saved: outputs/extractions_with_flags.csv")

# -----------------------------------------------------------------------------
# Part 7: Human Audit Sheet (Spot-Audits + Flagged Records)
# -----------------------------------------------------------------------------
# The audit sheet combines:
#   (a) A RANDOM sample of 5 docs (spot-audit — catches unexpected failures)
#   (b) ALL flagged docs (targeted review — catches known risk patterns)
#
# This ensures both random coverage and systematic follow-up on risky records.

audit_random  = extractions_df.sample(n=5, random_state=123)
audit_flagged = extractions_df[extractions_df["needs_human_review"]].copy()

audit_sheet = (
    pd.concat([audit_random, audit_flagged], ignore_index=True)
    .drop_duplicates(subset=["doc_id"])
    .sort_values("doc_id")
    .reset_index(drop=True)
)

# Blank columns for human annotators to fill in
audit_sheet["human_is_correct"]        = ""   # yes / no
audit_sheet["human_correct_event_type"] = ""  # corrected label if wrong
audit_sheet["human_correct_date_iso"]  = ""   # corrected date if wrong
audit_sheet["human_correct_location"]  = ""   # corrected location if wrong
audit_sheet["failure_mode"]            = ""   # e.g. date_missing, event_type_wrong
audit_sheet["reviewer_notes"]          = ""   # free text

audit_sheet.to_csv("outputs/human_audit_sheet.csv", index=False)

print("\n------------------------------")
print(f"Audit sheet: {len(audit_sheet)} rows ({len(audit_random)} random + flagged)")
print("Saved: outputs/human_audit_sheet.csv")
print("------------------------------")
print("Columns available for human review:")
print([c for c in audit_sheet.columns])

# -----------------------------------------------------------------------------
# Part 8: Precision / Recall Evaluation (Gold Set)
# -----------------------------------------------------------------------------
# We have 8 manually labelled documents (the "gold set").
# We compare the model's event_type predictions against the gold labels
# and compute a full classification report.

gold = pd.DataFrame([
    {"doc_id": "doc_001", "event_type_gold": "protest"},
    {"doc_id": "doc_002", "event_type_gold": "policy_change"},
    {"doc_id": "doc_003", "event_type_gold": "election"},
    {"doc_id": "doc_004", "event_type_gold": "disaster"},
    {"doc_id": "doc_005", "event_type_gold": "violence"},
    {"doc_id": "doc_006", "event_type_gold": "policy_change"},
    {"doc_id": "doc_007", "event_type_gold": "protest"},
    {"doc_id": "doc_008", "event_type_gold": "disaster"},
])

eval_df = gold.merge(
    extractions_df[["doc_id", "event_type"]],
    on="doc_id",
    how="left"
).rename(columns={"event_type": "event_type_pred"})

eval_df["event_type_pred"] = eval_df["event_type_pred"].fillna("MISSING")

print("\n------------------------------")
print("Evaluation table (gold vs predicted)")
print("------------------------------")
print(eval_df.to_string(index=False))

print("\n------------------------------")
print("Classification report (event_type)")
print("------------------------------")
print(classification_report(
    eval_df["event_type_gold"],
    eval_df["event_type_pred"],
    zero_division=0
))

# Accuracy summary
n_correct = (eval_df["event_type_gold"] == eval_df["event_type_pred"]).sum()
print(f"Overall accuracy on gold set: {n_correct}/{len(eval_df)} = {n_correct/len(eval_df):.1%}")

print("\n------------------------------")
print("Pipeline complete.")
print("Output files:")
print("  outputs/extractions_raw.csv        — raw model extractions")
print("  outputs/extractions_with_flags.csv — with uncertainty flags")
print("  outputs/human_audit_sheet.csv      — ready for human review")
print("------------------------------")

/Users/a75700/PycharmProjects/gazeextract/.venv

------------------------------
Review flag counts
------------------------------
flag_parse_failed        0
flag_low_confidence     15
flag_missing_date        0
flag_missing_country     0
flag_geo_unknown         6
needs_human_review      15
dtype: int64
Saved: outputs/extractions_with_flags.csv

------------------------------
Audit sheet: 15 rows (5 random + flagged)
Saved: outputs/human_audit_sheet.csv
------------------------------
Columns available for human review:
['doc_id', 'event_type', 'event_date_iso', 'date_is_approximate', 'country', 'admin1_or_state', 'city_or_local', 'geo_precision', 'actors', 'outcome_summary', 'extraction_confidence', 'raw_text', 'local_model_raw_output', 'parse_ok', 'parse_error', 'evidence_json', 'uncertainty_flags_json', 'flag_parse_failed', 'flag_low_confidence', 'flag_missing_date', 'flag_missing_country', 'flag_geo_unknown', 'needs_human_review', 'human_is_correct', 'human_correct_event_type', 'hum